# 02. Top-K Patterns

"Find the k largest / smallest / most frequent / closest" is the most common heap pattern.

**Learning order:** WHY A SIZE-K HEAP → KTH LARGEST → TOP K FREQUENT → K CLOSEST POINTS → LAST STONE → SORTED MATRIX → STREAM → INTERVIEW THINKING

## 1. The key trick: heap of size k, **opposite** type

To keep the **k largest** values, use a **min-heap of size k**. The root is the smallest of the k largest. When a new value is bigger than the root, it replaces it. Anything smaller can never be in the top k.

| Approach | Time | Space |
|---|---|---|
| sort everything | O(n log n) | O(1) or O(n) |
| **size-k heap** | **O(n log k)** | O(k) |
| quickselect | O(n) average | O(1) |

Real life: a "top 10 players" board that updates as scores arrive. You never store all scores.

In [1]:
import heapq
from collections import Counter

def kth_largest(nums, k):
    heap = []
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)                 # drop the smallest
    return heap[0]

print(kth_largest([3, 2, 1, 5, 6, 4], 2), kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4))   # 5 4

5 4


## 2. Top K Frequent Elements

Count with `Counter`, then take the `k` keys with the highest count. `heapq.nlargest(k, keys, key=count.get)` does this in O(n log k). (Bucket sort gives O(n), see Sorting.)

In [2]:
def top_k_frequent(nums, k):
    count = Counter(nums)
    return heapq.nlargest(k, count.keys(), key=count.get)

print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))   # [1, 2]

[1, 2]


## 3. K Closest Points to Origin

Distance `x² + y²` (no square root needed for comparison). Keep a **max-heap of size k** (negated distances): the root is the **farthest** among the k closest. Kick it out when a closer point arrives.

In [3]:
def k_closest(points, k):
    heap = []                                    # (-distance, point)
    for x, y in points:
        d = x * x + y * y
        heapq.heappush(heap, (-d, [x, y]))
        if len(heap) > k:
            heapq.heappop(heap)
    return [p for _, p in heap]

print(sorted(k_closest([[3, 3], [5, -1], [-2, 4]], 2)))   # [[-2, 4], [3, 3]]

[[-2, 4], [3, 3]]


## 4. Last Stone Weight (max-heap simulation)

Repeatedly smash the two heaviest stones. If different, push the difference back. Negate values to use `heapq` as a max-heap.

In [4]:
def last_stone_weight(stones):
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        a, b = -heapq.heappop(heap), -heapq.heappop(heap)
        if a != b:
            heapq.heappush(heap, -(a - b))
    return -heap[0] if heap else 0

print(last_stone_weight([2, 7, 4, 1, 8, 1]))   # 1

1


## 5. Kth Smallest in a Sorted Matrix

Rows and columns are sorted. Start with the **first element of each row** in a min-heap (as `(value, row, col)`). Pop the smallest `k` times; after popping `(v, r, c)` push the next element of that row `(r, c + 1)`.

This is the same idea as **merge k sorted lists**.

In [5]:
def kth_smallest_matrix(matrix, k):
    n = len(matrix)
    heap = [(matrix[r][0], r, 0) for r in range(n)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        val, r, c = heapq.heappop(heap)
        if c + 1 < n:
            heapq.heappush(heap, (matrix[r][c + 1], r, c + 1))
    return heap[0][0]

print(kth_smallest_matrix([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))   # 13

13


## 6. Kth Largest in a Data Stream (design)

Keep a **min-heap of size k**. `add(val)` pushes, pops if too big, and returns the root.

In [6]:
class KthLargest:
    def __init__(self, k, nums):
        self.k, self.heap = k, []
        for x in nums:
            self.add(x)

    def add(self, val):
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]

kl = KthLargest(3, [4, 5, 8, 2])
print([kl.add(x) for x in [3, 5, 10, 9, 4]])   # [4, 5, 5, 8, 8]

[4, 5, 5, 8, 8]


## 7. Interview thinking

**Signals:** "k largest/smallest", "top k", "k closest", "k-th in a stream", "k most frequent".

**Say:** "A heap of size k gives O(n log k) time and O(k) memory, which beats sorting when k is small. For a one-time query, quickselect is O(n) on average."

**Common mistakes:** using a max-heap of all n elements when a size-k min-heap is enough, wrong heap type (use **min-heap for k largest**), forgetting to negate for max-heap, computing square roots needlessly, not mentioning the alternatives.

## Quick Revision

- k largest → **min**-heap of size k. k smallest → **max**-heap of size k.
- O(n log k) time, O(k) space.
- Frequent: `Counter` + `nlargest`. Closest: squared distance.
- Sorted matrix / k lists: heap of `(value, row, col)`, push the next element of that row.
- Stream: keep the heap, answer is `heap[0]`.